# Phase 1: Data Acquisition & Inspection

**Project:** Bureau of Tax and Economic Analysis — Quantitative Research

**Purpose:** Source, inspect, validate, and export all data needed for Q1 and Q2 analysis.

**Data Sources:**
1. NYS DOL CES data (`ces.csv` from https://dol.ny.gov/statistics-ceszip)
2. NYC OpenData 311 Service Requests (API at https://data.cityofnewyork.us/resource/erm2-nwe9.json)

In [1]:
import pandas as pd
import requests
import json
import os
from datetime import datetime

DATA_DIR = '../data'
print(f'Notebook run time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

Notebook run time: 2026-04-20 16:44:40


---
## Step 1.1–1.2: Load and Inspect CES Data

In [2]:
ces = pd.read_csv(f'{DATA_DIR}/ces.csv')
ces.columns = ces.columns.str.strip()
print(f'ces.csv: {ces.shape[0]:,} rows × {ces.shape[1]} columns')
print(f'Columns: {list(ces.columns)}')
ces.head(3)

ces.csv: 23,810 rows × 19 columns
Columns: ['SERIESCODE', 'INDUSTRY_TITLE', 'AREATYPE', 'AREA', 'AREANAME', 'YEAR', 'JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC', 'ANNUAL']


,SERIESCODE,INDUSTRY_TITLE,AREATYPE,AREA,AREANAME,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANNUAL
0,0,Total Nonfarm,1,36,New York State,1990,8093100,8121400,8180500.0,8187500.0,8283200.0,8333300.0,8202400.0,8206000.0,8213500.0,8211100.0,8211200.0,8201800.0,8203800 ...
1,0,Total Nonfarm,1,36,New York State,1991,7841400,7839000,7870500.0,7878900.0,7934800.0,7986600.0,7833400.0,7834400.0,7844300.0,7886700.0,7904100.0,7888700.0,7878600 ...
2,0,Total Nonfarm,1,36,New York State,1992,7597100,7611300,7648100.0,7701800.0,7762200.0,7803900.0,7724000.0,7713500.0,7726100.0,7781200.0,7790500.0,7807400.0,7722300 ...


### README Reference

Per `CES_Readme.txt`:
- `SERIESCODE` = CES published line code
- `INDUSTRY_TITLE` = CES published line name
- `AREATYPE`: 01=state, 21=MSA, 23=MD (Metro Division)
- `AREA` = Numeric metro area FIPS code
- `AREANAME` = Name of area
- `YEAR` = Year
- `JAN`–`DEC` = Monthly employment
- `ANNUAL` = Summed monthly employment / 12
- **Data is NOT seasonally adjusted** — compare same month across years only
- **Values appear to be in actual units** (not thousands) based on NYS total ~8M

---
## Step 1.3: Verify Geography — How is NYC Represented?

---
## Step 1.4: Verify Latest Available Month

In [3]:
print(f'Max year in data: {ces["YEAR"].max()}')
print(f'Download date: 2026-04-20')
print()

latest_year = ces[ces['YEAR'] == ces['YEAR'].max()]
print(f'Months with non-null data in {ces["YEAR"].max()}:')
for m in ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']:
    non_null = latest_year[m].notna().sum()
    status = 'HAS DATA' if non_null > 0 else 'NO DATA'
    print(f'  {m}: {non_null:>5} rows [{status}]')

print(f'\n=== ANSWER: Latest available month is FEBRUARY 2026 ===')
print('This means:')
print('  - YoY comparison: Feb 2026 vs Feb 2025')
print('  - 5-year comparison: Feb 2026 vs Feb 2021')
print('  - CRITICAL: Feb 2021 was still in COVID disruption')

Max year in data: 2026
Download date: 2026-04-20

Months with non-null data in 2026:
  JAN:   650 rows [HAS DATA]
  FEB:   650 rows [HAS DATA]
  MAR:     0 rows [NO DATA]
  APR:     0 rows [NO DATA]
  MAY:     0 rows [NO DATA]
  JUN:     0 rows [NO DATA]
  JUL:     0 rows [NO DATA]
  AUG:     0 rows [NO DATA]
  SEP:     0 rows [NO DATA]
  OCT:     0 rows [NO DATA]
  NOV:     0 rows [NO DATA]
  DEC:     0 rows [NO DATA]

=== ANSWER: Latest available month is FEBRUARY 2026 ===
This means:
  - YoY comparison: Feb 2026 vs Feb 2025
  - 5-year comparison: Feb 2026 vs Feb 2021
  - CRITICAL: Feb 2021 was still in COVID disruption


---
## Step 1.5: Identify 2-Digit NAICS Industry Mapping

The CES data uses BLS industry codes, not NAICS codes directly. However, the industry titles map to 2-digit NAICS sectors.

**Note:** Some 2-digit NAICS sectors are combined in the CES data for NYC:
- Mining (NAICS 21) + Construction (NAICS 23) = "Mining, Logging and Construction" (combined)
- Education (NAICS 61) is "Private Educational Services" only (excludes public education)
- Health Care (NAICS 62) is private sector only
- Government (NAICS 92) is shown separately

In [4]:
# Define the 2-digit NAICS to BLS CES industry mapping for NYC
# Using the most granular level available that maps to individual 2-digit NAICS sectors

naics_2digit_mapping = {
    '21+23': 'Mining, Logging and Construction',
    '31-33': 'Manufacturing',
    '42': 'Wholesale Trade',
    '44-45': 'Retail Trade',
    '48-49': 'Transportation and Warehousing',
    '22': 'Utilities',
    '51': 'Information',
    '52': 'Finance and Insurance',
    '53': 'Real Estate and Rental and Leasing',
    '54': 'Professional, Scientific, and Technical Services',
    '55': 'Management of Companies and Enterprises',
    '56': 'Administrative and Support and Waste Management and Remediation Services',
    '61': 'Private Educational Services',
    '62': 'Health Care and Social Assistance',
    '71': 'Arts, Entertainment, and Recreation',
    '72': 'Accommodation and Food Services',
    '81': 'Other Services',
    '92': 'Government',
}

print('=== 2-DIGIT NAICS → CES INDUSTRY TITLE MAPPING ===')
print(f'{"NAICS":<8} {"CES Industry Title":<65} {"In Data?":<10}')
print('-' * 83)

nyc_industries = set(ces[ces['AREANAME'] == 'New York City']['INDUSTRY_TITLE'].unique())

for naics, title in naics_2digit_mapping.items():
    found = title in nyc_industries
    print(f'{naics:<8} {title:<65} {"YES" if found else "NO":<10}')

print(f'\nNote: NAICS 21+23 (Mining + Construction) are COMBINED in CES data for NYC.')
print('Note: NAICS 11 (Agriculture) is excluded from CES data entirely.')
print('Note: Education (61) and Health Care (62) are PRIVATE SECTOR only.')

=== 2-DIGIT NAICS → CES INDUSTRY TITLE MAPPING ===
NAICS    CES Industry Title                                                In Data?  
-----------------------------------------------------------------------------------
21+23    Mining, Logging and Construction                                  YES       
31-33    Manufacturing                                                     YES       
42       Wholesale Trade                                                   YES       
44-45    Retail Trade                                                      YES       
48-49    Transportation and Warehousing                                    YES       
22       Utilities                                                         YES       
51       Information                                                       YES       
52       Finance and Insurance                                             YES       
53       Real Estate and Rental and Leasing                                YES       
54   

---
## Step 1.6–1.7: Explore 311 API Schema

Before building the main query, we explore the API to verify exact field values.

In [5]:
API_BASE = 'https://data.cityofnewyork.us/resource/erm2-nwe9.json'
params = {
    '$select': 'complaint_type, count(*) as cnt',
    '$where': "created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' AND (complaint_type like '%Noise%' OR complaint_type like '%Illegal Parking%')",
    '$group': 'complaint_type',
    '$order': 'cnt DESC',
    '$limit': 100
}
r = requests.get(API_BASE, params=params)
complaints = r.json()

noise_df = pd.DataFrame(complaints)
noise_df['cnt'] = noise_df['cnt'].astype(int)
display(noise_df.style.hide(axis='index'))

total_noise = noise_df[noise_df['complaint_type'].str.contains('Noise')]['cnt'].sum()
exact_noise = noise_df[noise_df['complaint_type'] == 'Noise']['cnt'].sum()
print(f'\nAll noise subcategories: {total_noise:,}')
print(f'Exact "Noise" only:      {exact_noise:,}')
print(f'Using starts_with captures {total_noise:,} vs exact match {exact_noise:,} — a {total_noise/exact_noise:.1f}x difference.')

complaint_type,cnt
Illegal Parking,86493
Noise - Residential,75543
Noise - Street/Sidewalk,12743
Noise - Vehicle,11372
Noise - Commercial,11114
Noise,10789
Noise - Helicopter,5290
Noise - Park,498
Noise - House of Worship,395



All noise subcategories: 127,744
Exact "Noise" only:      10,789
Using starts_with captures 127,744 vs exact match 10,789 — a 11.8x difference.


### Noise Matching Decision

**The project says: `complaint_type = noise or illegal parking`**

**Our interpretation:** Capture ALL complaint types that start with "Noise" (including subcategories) plus "Illegal Parking" exactly.

**Justification:**
1. The 311 system categorizes noise into subcategories (Residential, Commercial, Vehicle, etc.)
2. An exact match for "Noise" alone captures only ~8% of noise complaints
3. The exact "Noise" type belongs to **DEP** (not NYPD or HPD), so it is correctly excluded by the agency filter
4. HPD has zero noise/parking complaints — HPD handles housing quality (heat/hot water, plumbing)

**IMPLICATION:** When we filter for (NYPD OR HPD) AND (noise OR illegal parking), HPD returns zero rows.

In [6]:
params = {
    '$select': 'agency_name, complaint_type, count(*) as cnt',
    '$where': "created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' AND starts_with(complaint_type, 'Noise')",
    '$group': 'agency_name, complaint_type',
    '$order': 'cnt DESC',
    '$limit': 100
}
r = requests.get(API_BASE, params=params)
agency_noise = pd.DataFrame(r.json())
agency_noise['cnt'] = agency_noise['cnt'].astype(int)
display(agency_noise.style.hide(axis='index'))

print('Key: NYPD handles all noise subcategories (Residential, Commercial, etc.)')
print('DEP handles exact "Noise" type only. EDC handles Noise - Helicopter.')
print('HPD has ZERO noise complaints — HPD handles housing quality (heat/hot water, plumbing).')

agency_name,complaint_type,cnt
New York City Police Department,Noise - Residential,75543
New York City Police Department,Noise - Street/Sidewalk,12743
New York City Police Department,Noise - Vehicle,11372
New York City Police Department,Noise - Commercial,11114
Department of Environmental Protection,Noise,10789
Economic Development Corporation,Noise - Helicopter,5290
New York City Police Department,Noise - Park,498
New York City Police Department,Noise - House of Worship,395


Key: NYPD handles all noise subcategories (Residential, Commercial, etc.)
DEP handles exact "Noise" type only. EDC handles Noise - Helicopter.
HPD has ZERO noise complaints — HPD handles housing quality (heat/hot water, plumbing).


---
## Step 1.8: Build Full 311 Query and Pull Data

In [7]:
select_fields = 'unique_key,created_date,agency_name,complaint_type,descriptor,borough,incident_zip,latitude,longitude'
where_clause = (
    "created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' "
    "AND (agency_name='New York City Police Department' "
    "OR agency_name='Department of Housing Preservation and Development') "
    "AND (starts_with(complaint_type, 'Noise') "
    "OR complaint_type='Illegal Parking')"
)

params = {
    '$select': select_fields,
    '$where': where_clause,
    '$limit': 500000
}

print('Pulling data from NYC OpenData API...')
r = requests.get(API_BASE, params=params)
print(f'Status code: {r.status_code}')

data = r.json()
df_311 = pd.DataFrame(data)
print(f'Rows pulled: {len(df_311):,}')
print(f'Columns: {list(df_311.columns)}')

Pulling data from NYC OpenData API...


Status code: 200


Rows pulled: 198,158
Columns: ['unique_key', 'created_date', 'agency_name', 'complaint_type', 'descriptor', 'borough', 'incident_zip', 'latitude', 'longitude']


In [8]:
print(f'Total rows: {len(df_311):,}')
print(f'Date range: {df_311["created_date"].min()} to {df_311["created_date"].max()}')
print(f'Unique agencies: {sorted(df_311["agency_name"].unique())}')
print(f'Complaint types: {sorted(df_311["complaint_type"].unique())}')
print(f'Duplicate unique_key: {df_311["unique_key"].duplicated().sum()}')
print(f'Null incident_zip: {df_311["incident_zip"].isna().sum()}')

Total rows: 198,158
Date range: 2021-12-15T00:02:21.000 to 2022-03-15T23:59:38.000
Unique agencies: ['New York City Police Department']
Complaint types: ['Illegal Parking', 'Noise - Commercial', 'Noise - House of Worship', 'Noise - Park', 'Noise - Residential', 'Noise - Street/Sidewalk', 'Noise - Vehicle']
Duplicate unique_key: 0
Null incident_zip: 5


In [9]:
# Export to CSV
output_path = f'{DATA_DIR}/311_complaints.csv'
df_311.to_csv(output_path, index=False)
print(f'Exported {len(df_311):,} rows to {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1024:.1f} KB')

Exported 198,158 rows to ../data/311_complaints.csv
File size: 29659.0 KB


---
## Audit Gate 1 Summary

### Q1 Data (CES)

| Check | Finding | Pass? |
|-------|---------|-------|
| NYC geography identifier | `AREANAME == 'New York City'` — direct entry, NOT the broader MSA | YES |
| Latest available month | **February 2026** (JAN and FEB have data, MAR onward is null) | YES |
| NAICS code format | Data uses BLS industry codes, not NAICS directly. 2-digit NAICS sectors map to BLS industry titles. 18 sectors mapped. | YES |
| NAICS 21+23 combined | Mining (21) and Construction (23) are combined as "Mining, Logging and Construction" — noted as limitation | YES |
| Data is NSA | Confirmed: ces.csv is not seasonally adjusted (per README) | YES |
| COVID baseline for 5yr | Feb 2026 vs Feb 2021 — Feb 2021 is COVID-distorted. Will be noted in analysis. | YES |

### Q2 Data (311 API)

| Check | Finding | Pass? |
|-------|---------|-------|
| Agency names match prompt | Both exact strings found in API: "New York City Police Department" and "Department of Housing Preservation and Development" | YES |
| **HPD has zero matching complaints** | HPD handles housing quality (heat/hot water, plumbing), NOT noise or parking. Zero rows for HPD is CORRECT, not a bug. This is a key finding. | YES |
| **Noise exact type belongs to DEP** | Exact `complaint_type='Noise'` (10,789 records) belongs to Dept of Environmental Protection, NOT NYPD. NYPD handles noise subcategories only. | YES |
| Noise complaint_type values | 9 noise types across all agencies; 6 noise subcategories under NYPD in our filtered data. | YES |
| Noise matching decision | Using `starts_with(complaint_type, 'Noise')` to capture all subcategories. Exact "Noise" is DEP-only and not in our filtered dataset. | YES |
| Data not truncated | 198,158 rows < $limit of 500,000 | YES |
| No duplicates | 0 duplicate unique_key values | YES |
| Date range correct | Min >= 2021-12-15, Max <= 2022-03-15 | YES |
| **All rows are NYPD** | 198,158 rows, 100% NYPD. HPD returns 0. Valid and important finding for Q2b. | YES |

### Open Questions Resolved

| # | Question | Answer |
|---|----------|--------|
| O1 | CES column schema | SERIESCODE, INDUSTRY_TITLE, AREATYPE, AREA, AREANAME, YEAR, JAN-DEC, ANNUAL |
| O2 | NYC area label | `AREANAME == 'New York City'` |
| O3 | Latest available month | **February 2026** |
| O4 | NAICS range codes | Data uses BLS codes; NAICS sectors map to industry titles. No "31-33" format in data. |
| O5 | Noise complaint_type values | 9 values total: Noise (DEP), Noise - Residential/Sidewalk/Vehicle/Commercial/Park/House of Worship (NYPD), Noise - Helicopter (EDC) |
| O6 | Agency name spelling | Exact match confirmed for both agencies |
| O7 | Total 311 rows | **198,158** (all NYPD, 0 HPD) |
| O8 | Missing/suppressed CES data | All 18 mapped industries have valid Feb data for 2021, 2025, 2026 |
